# Veri Ön İşleme

In [66]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)

## 1. Veri Yükleme

In [67]:
df = pd.read_csv('../data/processed/merged.csv')
print(df.shape)
df.head()

(39226, 18)


,player_id,position,sub_position,foot,height_in_cm,country_of_citizenship,market_value_in_eur,player_club_domestic_competition_id,total_goals,total_assists,total_minutes,total_matches,total_yellow_cards,total_red_cards,goals_per_match,assists_per_match,minutes_per_match,age
0,10,Attack,Centre-Forward,right,184.0,Germany,1000000,IT1,48.0,25.0,8808.0,136.0,19.0,0.0,0.352941,0.183824,64.764706,47.0
1,26,Goalkeeper,Goalkeeper,left,190.0,Germany,750000,L1,0.0,0.0,13508.0,152.0,4.0,2.0,0.000000,0.000000,88.868421,45.0
2,65,Attack,Centre-Forward,NaN,NaN,Bulgaria,1000000,GR1,38.0,13.0,8788.0,122.0,11.0,1.0,0.311475,0.106557,72.032787,45.0
3,77,Defender,Centre-Back,NaN,NaN,Brazil,200000,IT1,0.0,0.0,307.0,4.0,0.0,0.0,0.000000,0.000000,76.750000,47.0
4,80,Goalkeeper,Goalkeeper,right,194.0,Germany,100000,L1,0.0,0.0,1080.0,12.0,0.0,0.0,0.000000,0.000000,90.000000,45.0


## 2. Eksik Veri Temizleme

In [68]:
# maç istatistiği ve yaşı olmayanları düşürme
df = df.dropna(subset=['total_goals', 'total_assists', 'total_minutes', 'age'])

# kalan eksikleri doldurma
df['foot'] = df['foot'].fillna('unknown')
df['height_in_cm'] = df['height_in_cm'].fillna(df['height_in_cm'].mean())
df['sub_position'] = df['sub_position'].fillna(df['position'])
df['country_of_citizenship'] = df['country_of_citizenship'].fillna('unknown')
df['player_club_domestic_competition_id'] = df['player_club_domestic_competition_id'].fillna('unknown')

# geçersiz pozisyon değerlerini düşürme
df = df[df['position'] != 'Missing']

print("Eksik veri kontrolü:")
print(df.isnull().sum())
print("Eksik veri temizlendikten sonra veri boyutu:", df.shape)

Eksik veri kontrolü:
player_id                              0
position                               0
sub_position                           0
foot                                   0
height_in_cm                           0
country_of_citizenship                 0
market_value_in_eur                    0
player_club_domestic_competition_id    0
total_goals                            0
total_assists                          0
total_minutes                          0
total_matches                          0
total_yellow_cards                     0
total_red_cards                        0
goals_per_match                        0
assists_per_match                      0
minutes_per_match                      0
age                                    0
dtype: int64
Eksik veri temizlendikten sonra veri boyutu: (27558, 18)


## 3. Aykırı Değer Temizleme

In [69]:
# boy için mantıksız değerleri düşürme (150 cm altı)
print("150 cm altı futbolcu sayısı:", (df['height_in_cm'] < 150).sum())
df = df[df['height_in_cm'] >= 150]

# yaş için mantıksız değerleri düşürme (15 altı ve 45 üstü)
print("15 altı yaş:", (df['age'] < 15).sum())
print("45 üstü yaş:", (df['age'] > 45).sum())
df = df[(df['age'] >= 15) & (df['age'] <= 45)]

print("Aykırı değer temizlendikten sonra veri boyutu:", df.shape)

150 cm altı futbolcu sayısı: 3
15 altı yaş: 0
45 üstü yaş: 565
Aykırı değer temizlendikten sonra veri boyutu: (26990, 18)


## 4. Kapsam Filtreleme

In [70]:
# emekli futbolcuları düşürme (38 yaş ve üstü)
# model aktif transfer piyasasına odaklanmakta
print("38 yaş ve üstü futbolcu sayısı:", (df["age"] >= 38).sum())
df = df[df["age"] < 38]

# sembolik piyasa değerli oyuncuları düşürme (100.000€ altı)
print("100k€ altı piyasa değeri:", (df["market_value_in_eur"] < 100_000).sum())
df = df[df["market_value_in_eur"] >= 100_000]

print("Kapsam filtrelemesi sonrası veri boyutu:", df.shape)


38 yaş ve üstü futbolcu sayısı: 4759
100k€ altı piyasa değeri: 2090
Kapsam filtrelemesi sonrası veri boyutu: (20141, 18)


## 5. Özellik Mühendisliği

In [71]:
# hedef değişkene log dönüşümü uygulama
df['log_market_value'] = np.log1p(df['market_value_in_eur'])

# çoklu doğrusal bağlantı nedeniyle total_minutes düşürme
df = df.drop(columns=['total_minutes'])

print("Yeni sütunlar:", df.columns.tolist())
print("Özellik mühendisliğinden sonra veri boyutu:", df.shape)

Yeni sütunlar: ['player_id', 'position', 'sub_position', 'foot', 'height_in_cm', 'country_of_citizenship', 'market_value_in_eur', 'player_club_domestic_competition_id', 'total_goals', 'total_assists', 'total_matches', 'total_yellow_cards', 'total_red_cards', 'goals_per_match', 'assists_per_match', 'minutes_per_match', 'age', 'log_market_value']
Özellik mühendisliğinden sonra veri boyutu: (20141, 18)


## 6. Kodlama

In [72]:
# kategorik sütunları kodlama
categorical_cols = ['position', 'sub_position', 'foot', 'country_of_citizenship', 'player_club_domestic_competition_id']

le = LabelEncoder()
for col in categorical_cols:
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))

print("Kodlanan sütunlar:", categorical_cols)
print("Veri boyutu:", df.shape)

Kodlanan sütunlar: ['position', 'sub_position', 'foot', 'country_of_citizenship', 'player_club_domestic_competition_id']
Veri boyutu: (20141, 23)


## 7. Ölçekleme

In [73]:
# sayısal sütunları ölçekleme
numeric_cols = ['age', 'height_in_cm', 'total_goals', 'total_assists', 
                'total_matches', 'total_yellow_cards', 'total_red_cards',
                'goals_per_match', 'assists_per_match', 'minutes_per_match']

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("Ölçekleme tamamlandı.")
print(df_scaled[numeric_cols].describe().round(2))

Ölçekleme tamamlandı.
            age  height_in_cm  total_goals  total_assists  total_matches  \
count  20141.00      20141.00     20141.00       20141.00       20141.00   
mean      -0.00          0.00        -0.00           0.00          -0.00   
std        1.00          1.00         1.00           1.00           1.00   
min       -2.55         -3.63        -0.39          -0.45          -0.76   
25%       -0.72         -0.64        -0.39          -0.45          -0.68   
50%        0.09         -0.01        -0.33          -0.37          -0.43   
75%        0.91          0.71        -0.04          -0.03           0.30   
max        1.72          3.56        29.96          18.15           6.52   

       total_yellow_cards  total_red_cards  goals_per_match  \
count            20141.00         20141.00         20141.00   
mean                -0.00            -0.00             0.00   
std                  1.00             1.00             1.00   
min                 -0.65            -0.4

## 8. Özellik Seçimi

In [74]:
# gereksiz sütunları düşürme
drop_cols = ['player_id', 'position', 'sub_position', 'foot', 
             'country_of_citizenship', 'player_club_domestic_competition_id',
             'market_value_in_eur']
df_scaled = df_scaled.drop(columns=drop_cols)

# sütunları anlamlı bir şekilde sıralama
ordered_cols = [
    "log_market_value",
    "age", "height_in_cm",
    "total_goals", "total_assists", "total_matches",
    "total_yellow_cards", "total_red_cards",
    "goals_per_match", "assists_per_match", "minutes_per_match",
    "position_encoded", "sub_position_encoded", "foot_encoded",
    "country_of_citizenship_encoded", "player_club_domestic_competition_id_encoded"
]
df_scaled = df_scaled[ordered_cols]

print("Modele girecek değişkenler:", df_scaled.columns.tolist())

Modele girecek değişkenler: ['log_market_value', 'age', 'height_in_cm', 'total_goals', 'total_assists', 'total_matches', 'total_yellow_cards', 'total_red_cards', 'goals_per_match', 'assists_per_match', 'minutes_per_match', 'position_encoded', 'sub_position_encoded', 'foot_encoded', 'country_of_citizenship_encoded', 'player_club_domestic_competition_id_encoded']


## 9. Bulgular
- Maç istatistiği bulunmayan 11585 futbolcu veri setinden çıkarılmıştır.
- Boy (150 cm altı), yaş (45 üstü) ve diğer aykırı değerler temizlenmiştir.
- 38 yaş ve üstü futbolcular emekli/aktif piyasa dışı kabul edilerek elenmiştir; model aktif transfer piyasasına odaklanmaktadır.
- 100.000€ altı piyasa değerine sahip sembolik değerli oyuncular kapsam dışı bırakılmıştır.
- Eksik kategorik değerler `unknown` ile doldurulmuş, sayısal eksikler ortalama ile tamamlanmıştır.
- Piyasa değerinin sağa çarpık dağılımı nedeniyle log dönüşümü uygulanmıştır.
- Toplam süre (total_minutes) ile toplam maç sayısı (total_matches) arasındaki 0.98 korelasyon nedeniyle total_minutes çıkarılmıştır.
- Kategorik değişkenler sayısal kodlamaya, sayısal değişkenler StandardScaler ile ölçeklemeye tabi tutulmuştur.
- Ön işleme sonrası 20141 futbolcu ve 16 değişken ile modelleme aşamasına geçilecektir.

## 10. Veri Kaydetme

In [75]:
# işlenmiş veriyi kaydetme
df_scaled.to_csv('../data/processed/processed.csv', index=False)
print("Veri ön işleme sonucu veri boyutu:", df_scaled.shape)

Veri ön işleme sonucu veri boyutu: (20141, 16)
